# 🚀 SSL400 Training Notebook - Fully Automated

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# Cell 2 — COPY PROJECT TO LOCAL GPU DISK (For maximum speed!)
import shutil
import os
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/SSL400_Research'
LOCAL_PROJECT_PATH = '/content/ssl400'

print('Copying code and config from Google Drive to local Colab memory...')
os.makedirs(LOCAL_PROJECT_PATH, exist_ok=True)
if os.path.exists(os.path.join(DRIVE_PROJECT_PATH, 'config.yaml')):
    shutil.copy2(os.path.join(DRIVE_PROJECT_PATH, 'config.yaml'), LOCAL_PROJECT_PATH)

local_src = os.path.join(LOCAL_PROJECT_PATH, 'src')
if os.path.exists(local_src):
    shutil.rmtree(local_src)
shutil.copytree(os.path.join(DRIVE_PROJECT_PATH, 'src'), local_src)

local_models = os.path.join(LOCAL_PROJECT_PATH, 'models')
if os.path.exists(local_models):
    shutil.rmtree(local_models)
if os.path.exists(os.path.join(DRIVE_PROJECT_PATH, 'models')):
    shutil.copytree(os.path.join(DRIVE_PROJECT_PATH, 'models'), local_models)

print('✅ Code successfully copied to /content/ssl400!')
os.chdir(LOCAL_PROJECT_PATH)
print(f'📁 Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — INSTALL DEPENDENCIES
!pip install tf-keras tf-models-official --quiet
print('✅ Dependencies installed')

In [ ]:
# Cell 4 — EXPERIMENT CONFIGURATION
EXP_ID = 2              # Experiment 2 (CLAHE + Gamma Correction)
BATCH_SIZE = 8          # Use 4 if GPU runs out of memory
DRIVE_MODEL_DIR = f'/content/drive/MyDrive/SSL400_Research/models/experiment_{EXP_ID}'

# RESUME SUPPORT ENABLED
print('✅ Ready to resume training from latest checkpoint!')

In [ ]:
# Cell 5 — RUN TRAINING
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!python /content/ssl400/src/training/train.py --exp_id={EXP_ID} --batch_size={BATCH_SIZE} --drive_dir={DRIVE_MODEL_DIR}

In [ ]:
# Cell 6 — EVALUATE MODEL ON TEST SET (Precision, Recall, F1)
!python /content/ssl400/src/evaluation/evaluate.py --exp_id={EXP_ID}

In [ ]:
# Cell 7 — SHOW TRAINING CURVES
import pandas as pd
import matplotlib.pyplot as plt
import os
log_file = f'/content/ssl400/logs/experiment_{EXP_ID}/training_log_phase2.csv'
if os.path.exists(log_file):
    df = pd.read_csv(log_file)
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(df['accuracy'], label='Train Acc', linewidth=2, color='#2ca02c')
    ax1.plot(df['val_accuracy'], label='Val Acc', linewidth=2, color='#d62728', linestyle='--')
    ax1.set_title(f'Accuracy Curve - Experiment {EXP_ID}', fontsize=14, pad=15)
    ax1.legend()
    ax2.plot(df['loss'], label='Train Loss', linewidth=2, color='#1f77b4')
    ax2.plot(df['val_loss'], label='Val Loss', linewidth=2, color='#ff7f0e', linestyle='--')
    ax2.set_title(f'Loss Curve - Experiment {EXP_ID}', fontsize=14, pad=15)
    ax2.legend()
    plt.show()
else:
    print('Training has not finished yet, so there is no graph to draw!')
